<a href="https://colab.research.google.com/github/Sanika515/AIML/blob/main/3_Implement_an_RNN_to_generate_text_based_on_a_corpus_of_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Load a text corpus (Shakespeare). To use your own text: text = open('myfile.txt').read()
url = 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
path = keras.utils.get_file('shakespeare.txt', origin=url)
text = open(path, 'rb').read().decode('utf-8')[:150000]      # first 100k characters (fast)
print("Characters:", len(text)); print(text[:250])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Characters: 150000
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



In [ ]:
# 2. Character vocabulary: every unique character -> a number
chars = sorted(set(text)); vocab = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)
print("Vocabulary size:", vocab)

# 3. Make training samples: 40 characters in -> the NEXT character out
SEQ, STEP = 40, 2
X, y = [], []
for i in range(0, len(text) - SEQ, STEP):
    X.append([char2idx[c] for c in text[i:i + SEQ]])
    y.append(char2idx[text[i + SEQ]])
X, y = np.array(X), np.array(y)
print("X:", X.shape, "y:", y.shape)

Vocabulary size: 61
X: (74980, 40) y: (74980,)


In [ ]:
# 4. Build the model
model = keras.Sequential([
    layers.Input(shape=(SEQ,)),
    layers.Embedding(vocab, 32),                 # char id -> 32-number vector
    layers.SimpleRNN(256),
    layers.Dense(vocab, activation='softmax')    # probability of each next character
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 32)         │         1,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 256)            │        73,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 61)             │        15,677 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 91,613 (357.86 KB)

 Trainable params: 91,613 (357.86 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 5. Train (loss should fall steadily)
history = model.fit(X, y, batch_size=128, epochs=15)

Epoch 1/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 30s 49ms/step - loss: 2.6227
Epoch 2/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 28s 48ms/step - loss: 2.1574
Epoch 3/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 42s 49ms/step - loss: 2.0009
Epoch 4/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 28s 48ms/step - loss: 1.8891
Epoch 5/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 28s 48ms/step - loss: 1.8045
Epoch 6/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 29s 49ms/step - loss: 1.7332
Epoch 7/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 40s 48ms/step - loss: 1.6739
Epoch 8/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 41s 48ms/step - loss: 1.6262
Epoch 9/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 28s 48ms/step - loss: 1.5801
Epoch 10/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 29s 49ms/step - loss: 1.5434
Epoch 11/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 28s 48ms/step - loss: 1.5066
Epoch 12/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 29s 49ms/step - loss: 1.4745
Epoch 13/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 28s 48ms/step - loss: 1.4443
Epoch 14/15
586/586 ━━━━━━━━━━━━━━━━━━━━ 41s 49ms/step - loss: 1.4150
Epoch 15/15
586/586 ━━━━━━━━━

In [ ]:
# 6. Generate new text, one character at a time
def sample(probs, temperature=0.7):
    probs = np.log(np.asarray(probs, dtype='float64') + 1e-9) / temperature
    probs = np.exp(probs); probs /= probs.sum()
    return np.random.choice(len(probs), p=probs)

def generate(seed, length=300, temperature=0.7):
    out = seed.rjust(SEQ)                                   # pad seed to 40 chars
    for _ in range(length):
        ids = np.array([[char2idx.get(c, 0) for c in out[-SEQ:]]])
        probs = model(ids, training=False).numpy()[0]
        out += idx2char[sample(probs, temperature)]
    return out.strip()

print(generate("ROMEO:", 300, 0.5))
print("\n---- higher temperature = more random ----")
print(generate("ROMEO:", 200, 1.2))

ROMEO:
SI Prithee,
When, be the commons of the marked and coust, the city and the mult the common follows to the common fools of your chowes all under to mund the people, and the gods for the morten the common fought with the end and strection the many thing the Cilitius
Hath all the compon that have been

---- higher temperature = more random ----
ROMEO:, 'y,
Mestive deatleint the rany, th shere have never ampition.

All:
Why, May mankenon'd abuntart: from, as lengees:
Down dom. O' their tange;
I consprither,
Which whats protith:
And not the mil.

Th


In [ ]:
1. What is the problem?
We give the computer a large piece of text (a corpus), here Shakespeare's writing. It must learn the style of the writing and then produce new text on its own, one character at a time.

2. What is the solution?
We use an RNN (Recurrent Neural Network). Text is a sequence: the order of letters matters ("cat" is not "act"). A normal network has no memory of what came before. An RNN does: it reads one character at a time and keeps a hidden state (its memory) that it passes to the next step.
Analogy: a phone's autocomplete. You type "Good mor", and it guesses "n". Our model does the same, but it learns from Shakespeare. Predict a character, add it to the text, predict the next one, and repeat. That repeating process is how a whole paragraph gets generated.
The core idea:"Given the previous 40 characters, predict the next character."

RNN equation (examiners love this):

h(t) = tanh( Wx · x(t) + Wh · h(t−1) + b )

x(t) is the current character, and h(t−1) is the memory from the previous step.
h(t) is the new memory. The same weights are reused at every time step.

################## The code ###############

Step 1: Load the text
url = 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
path = keras.utils.get_file('shakespeare.txt', origin=url)
text = open(path, 'rb').read().decode('utf-8')[:100000]

Downloads Shakespeare and keeps the first 100,000 characters, so training stays fast. (You can use any .txt file of your own.)

Step 2: Build the vocabulary

python
chars = sorted(set(text)); vocab = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)

Computers only understand numbers. set(text) finds every unique character (letters, space, punctuation). char2idx maps each character to a number (like 'a' → 39). idx2char maps back. Vocabulary size is the number of unique characters.

Step 3: Make training samples

python
SEQ, STEP = 40, 3
for i in range(0, len(text) - SEQ, STEP):
    X.append([char2idx[c] for c in text[i:i + SEQ]])
    y.append(char2idx[text[i + SEQ]])

A window of 40 characters slides over the text. For each window, X is the 40 characters and y is the next character.
Example: X = "To be, or not to be, that is the questi", y = "o".
STEP = 3 means the window moves 3 characters at a time, to avoid too much overlap.

Step 4: Build the model

python
model = keras.Sequential([
    layers.Input(shape=(SEQ,)),
    layers.Embedding(vocab, 32),
    layers.SimpleRNN(128),
    layers.Dense(vocab, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
Embedding turns each character number into a vector of 32 numbers. This is better than one-hot encoding because it is small and the model learns which characters behave alike.
SimpleRNN(128) has 128 memory units. It reads all 40 characters and gives its last memory.
Dense + softmax gives a probability for each possible next character.
The loss is sparse categorical cross-entropy because the target y is a single number (the correct character).

Step 5: Train

python
history = model.fit(X, y, batch_size=128, epochs=15)

The model sees about 33,000 samples 15 times. The loss should keep decreasing.

Step 6: Generate new text

python
def sample(probs, temperature=0.7):
    probs = np.log(probs + 1e-9) / temperature
    probs = np.exp(probs); probs /= probs.sum()
    return np.random.choice(len(probs), p=probs)

def generate(seed, length=300, temperature=0.7):
    out = seed.rjust(SEQ)
    for _ in range(length):
        ids = np.array([[char2idx.get(c, 0) for c in out[-SEQ:]]])
        probs = model(ids, training=False).numpy()[0]
        out += idx2char[sample(probs, temperature)]
    return out.strip()

print(generate("ROMEO:", 300, 0.7))

How generation works:
Start with a seed ("ROMEO:"), padded to 40 characters.
The model predicts probabilities for the next character.
We pick one character using those probabilities.
Add it to the text, drop the oldest character, and repeat 300 times.

What is temperature? It controls how adventurous the picking is.

Low (0.7): the model plays safe and picks the most likely characters. The text is more correct but repetitive.
High (1.2): the model takes risks, so the text is more creative but messier, with more spelling errors.
4. The output you will see
First 250 characters of Shakespeare, and the character count (100,000).
Vocabulary size: about 60–65, and X: (~33300, 40), y: (~33300,).
Model summary: Embedding about 2,000 parameters, SimpleRNN 20,608, Dense about 8,000.
Training log: 15 epochs. The loss falls steadily, roughly from about 3 to below 2.
Generated text for ROMEO:. It looks like Shakespeare in shape (line breaks, capital names, colons) with some real words like "the", "and", "my", mixed with some made-up words. That is normal for a SimpleRNN, because it has weak memory and little training.
A second generated text at temperature 1.2, which looks messier.

Don't worry if the text isn't perfect. Say: "The model learned the structure and style at character level. It improves with more data, more epochs, or an LSTM."

#####BLOCK ############
           START
             |
             v
       Load Shakespeare Text
             |
             v
       Character Vocabulary
       (Unique Characters)
             |
             v
       Character Encoding
        Character → Number
             |
             v
       Create Sequences
       Input: 40 Characters
       Output: Next Character
             |
             v
          RNN MODEL
             |
             v
          Embedding
             |
             v
         SimpleRNN
          (128 Units)
             |
             v
        Dense + Softmax
             |
             v
         Train Model
             |
             v
       Provide Seed Text
             |
             v
       Predict Next Character
             |
             v
       Append Predicted Char
             |
             v
        Repeat Generation
             |
             v
       Generated Shakespeare
             |
             v
            STOP


##### QUE ###########
6. Questions the examiner may ask
What is an RNN? A network with a loop. The hidden state from the previous step is fed into the next step, so it has memory of earlier inputs. It suits sequences such as text, speech and time series.
Why not a normal neural network for text? It treats every input independently and has no memory of order.
What is the vanishing gradient problem? During backpropagation through time, gradients are multiplied many times by small numbers and shrink towards zero. Early characters stop influencing learning, so the RNN forgets long-range information. The LSTM solves this (Problem 4).
What is BPTT? Backpropagation Through Time. We unroll the RNN across the time steps and apply backpropagation through all of them.
Why an Embedding layer? It turns a character number into a small, learned vector, which is more efficient and meaningful than one-hot encoding.
What is temperature? It controls the randomness of sampling. Low is safe and repetitive, high is creative and messy.
Why softmax at the end? We are choosing one out of about 65 possible next characters, so we need probabilities that add up to 1.
Character-level vs word-level model? Character-level has a tiny vocabulary and no unknown words, but needs longer sequences. Word-level captures meaning better but needs a huge vocabulary.
Why does the generated text have mistakes? Small model, short training, only 100,000 characters, and the SimpleRNN's short mem